# Лабораторная работа №3. Транскрипция

В работе использован корпус RUSLAN и фонемный инвентарь Montreal Forced Aligner.

In [72]:
import csv
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display
from praatio import textgrid
from tqdm.auto import tqdm

LAB_DIR = Path.cwd()
PROJECT_ROOT = LAB_DIR.parents[1]
ALIGN_DIR = PROJECT_ROOT / 'data' / 'RUSLAN_22050_align_v2'
TOKENS_PATH = LAB_DIR / 'data' / 'tokens_phones.csv'
TOKENS_SIL_PATH = LAB_DIR / 'data' / 'tokens_phones_sil.csv'
LEXICON_PATH = LAB_DIR / 'data' / 'lexicon.csv'
MFA_PATH = LAB_DIR / 'data' / 'russian_mfa.dict'
TARGET_PATH = PROJECT_ROOT / 'common' / 'phones_mfa.txt'
RAW_META_PATH = PROJECT_ROOT / 'data' / 'metadata_RUSLAN_22200.csv'
NORMALIZED_META_PATH = PROJECT_ROOT / 'data' / 'metadata_RUSLAN_22200_normalized.csv'
SIL = '<SIL>'

tokens = pd.read_csv(TOKENS_PATH, sep='|')
tokens_sil = pd.read_csv(TOKENS_SIL_PATH, sep='|')
lexicon = pd.read_csv(LEXICON_PATH, sep='|')

## 1. Подготовка данных

In [73]:
summary = pd.Series({
    'Фразы': tokens.id.nunique(),
    'Фразы train': tokens.loc[tokens.set == 'train', 'id'].nunique(),
    'Фразы test': tokens.loc[tokens.set == 'test', 'id'].nunique(),
    'Слова': len(tokens),
    'Паузы': tokens_sil.token.eq(SIL).sum(),
    'Словоформы в train': lexicon.token.nunique(),
})
summary.to_frame('Количество')

,Количество
Фразы,19649
Фразы train,15680
Фразы test,3969
Слова,219419
Паузы,52650
Словоформы в train,38574


## 2. Фонемный инвентарь

Обозначения MFA основаны на IPA. Диакритики `ʲ` и `̪` обозначают палатализацию и зубное место образования. Знак долготы удалён при подготовке данных, поскольку его нет в целевом списке.

In [74]:
kept_ids = set(tokens.id)
duration_rows = []
for path in tqdm(sorted(ALIGN_DIR.glob('*.TextGrid')), desc='TextGrid'):
    if path.stem not in kept_ids:
        continue
    grid = textgrid.openTextgrid(str(path), includeEmptyIntervals=True)
    for entry in grid.getTier('phones').entries:
        phone = entry.label.replace('ː', '')
        if phone and phone != 'spn':
            duration_rows.append((phone, entry.end - entry.start))

durations = pd.DataFrame(duration_rows, columns=['phone', 'duration'])
phone_stats = durations.groupby('phone').duration.agg(['count', 'mean', 'std', 'median'])
phone_stats.sort_values('count', ascending=False).head(15).round(4)

TextGrid: 100%|██████████| 22199/22199 [00:09<00:00, 2301.91it/s]


,count,mean,std,median
phone,,,,
ɪ,100473,0.0177,0.0257,0.01
ə,93904,0.0398,0.0347,0.03
ɐ,65387,0.0642,0.0210,0.06
a,55338,0.0890,0.0358,0.09
t̪,54318,0.0789,0.0321,0.07
j,48364,0.0656,0.0391,0.05
n̪,48356,0.0684,0.0315,0.06
e,43388,0.0737,0.0350,0.07
s̪,41264,0.1019,0.0335,0.10


In [75]:
target = {line.strip() for line in TARGET_PATH.read_text(encoding='utf-8').splitlines() if line.strip()}
observed = set(phone_stats.index)
display(phone_stats.sort_values('count').head(12).round(4))
pd.Series({
    'Фонем в корпусе': len(observed),
    'Вне целевого списка': sorted(observed - target),
    'Не встретились в корпусе': sorted(target - observed - {SIL}),
}).to_frame('Значение')

,count,mean,std,median
phone,,,,
d̪z̪,1,0.0600,NaN,0.060
ʑ,19,0.1300,0.0249,0.120
ɣ,20,0.0555,0.0282,0.045
dʐ,225,0.1512,0.0313,0.150
tʂ,277,0.1468,0.0301,0.150
ç,496,0.1427,0.0323,0.140
ɵ,946,0.0853,0.0503,0.070
fʲ,1873,0.1075,0.0333,0.110
ʉ,2400,0.0387,0.0333,0.030


,Значение
Фонем в корпусе,55
Вне целевого списка,[]
Не встретились в корпусе,[]


In [76]:
mfa_words = set()
with open(MFA_PATH, encoding='utf-8') as file:
    for line in file:
        parts = line.rstrip().split('\t')
        if len(parts) >= 2 and 'spn' not in parts[-1].split():
            mfa_words.add(parts[0].lower())

test = tokens[tokens.set == 'test'].copy()
corpus_words = set(lexicon.token)
test['source'] = test.token.map(
    lambda word: 'Корпус' if word in corpus_words else ('MFA' if word in mfa_words else 'Модель')
)
coverage = test.source.value_counts().rename('Токены').to_frame()
coverage['Доля'] = coverage['Токены'] / len(test)
coverage

,Токены,Доля
source,,
Корпус,38537,0.870165
MFA,4506,0.101745
Модель,1244,0.028090


In [77]:
variants = lexicon.groupby('token').agg(
    pronunciations=('phones', 'nunique'),
    frequency=('frequency', 'sum'),
).query('pronunciations > 1').sort_values(['frequency', 'pronunciations'], ascending=False)
display(variants.head(10))
lexicon[lexicon.token.isin(['замок', 'все', 'всё'])].sort_values(['token', 'frequency'], ascending=[True, False])

,pronunciations,frequency
token,,
в,5,4532
не,3,2932
на,2,2375
что,2,1959
с,6,1582
а,3,1423
к,3,949
все,2,931
из,4,740


,token,phones,frequency
4263,все,f sʲ ɵ,573
4264,все,f sʲ e,358
9684,замок,z̪ a m ə k,2
9685,замок,z̪ ɐ m o k,1


In [78]:
raw = pd.read_csv(RAW_META_PATH, sep='|', header=None, names=['id', 'text'], quoting=csv.QUOTE_NONE)
kept = set(pd.read_csv(NORMALIZED_META_PATH, sep='|', header=None, usecols=[0])[0])
removed = raw[~raw.id.isin(kept)]
removed[removed.text.str.contains(r'\d|[А-ЯЁ]{2,}', regex=True, na=False)].head(10)

,id,text
662,000662_RUSLAN,РЯДОМ С ГЕЙНЕ
748,000748_RUSLAN,О РАССКАЗАХ С. ДОВЛАТОВА
994,000994_RUSLAN,Отдел культуры Ленинградского ОК КПСС
996,000996_RUSLAN,Ленинградский ОК вээлксэм
1595,001595_RUSLAN,СЕМЬ НОТ В ТИШИНЕ
1638,001638_RUSLAN,ОБСУЖДЕНИЕ В цека
1766,001766_RUSLAN,Группа эстонцев направила петицию в ООН.
2173,002173_RUSLAN,До этого рукопись была положительно отрецензирована доцентом ТГУ В. Беззубовым.
3015,003015_RUSLAN,ТЕЛЕЖКА С ХЛЕБНЫМ КВАСОМ
3499,003499_RUSLAN,"Просто мы не знали, как это делается в США."


## 3. G2P

Транскриптор сначала ищет слово в словаре train, затем в словаре MFA. Для остатка он использует модель символьных окон, обученную на train. Модель выравнивает буквы и фонемы, запоминает варианты с учётом соседних букв, окончания и позиции гласной.

In [79]:
from g2p import G2P, build_lexicon, calc_metrics, load_mfa_dictionary, phone_error_rate

train = tokens[tokens.set == 'train']
test = tokens[tokens.set == 'test']
corpus_lexicon = build_lexicon(train)
mfa_lexicon = load_mfa_dictionary()
g2p = G2P(corpus_lexicon, mfa_lexicon)
metrics = pd.DataFrame(calc_metrics(g2p, test, corpus_lexicon, mfa_lexicon)).T
metrics

100%|██████████| 3969/3969 [00:00<00:00, 4519.14it/s]


,tokens,per,word_accuracy
text,3969.0,0.013472,0.560091
corpus,38537.0,0.010384,0.957314
mfa,4506.0,0.004945,0.979805
model,1244.0,0.091592,0.442926


In [80]:
def evaluate_stages(frame):
    reference = [value.split() for value in frame.phones]
    corpus_only, corpus_mfa, full = [], [], []
    for token in frame.token:
        corpus_prediction = g2p._lookup(token, corpus_lexicon)
        mfa_prediction = corpus_prediction or g2p._lookup(token, mfa_lexicon)
        corpus_only.append(corpus_prediction or [])
        corpus_mfa.append(mfa_prediction or [])
        full.append(g2p.transcribe([token])[0])
    return {
        'Словарь train': phone_error_rate(reference, corpus_only),
        'Словарь train и MFA': phone_error_rate(reference, corpus_mfa),
        'Полная модель': phone_error_rate(reference, full),
    }

pd.DataFrame({'train': evaluate_stages(train), 'test': evaluate_stages(test)})

,train,test
Словарь train,0.007481,0.209614
Словарь train и MFA,0.007481,0.057327
Полная модель,0.007481,0.013472


## 4. Метаданные для акустической модели

In [81]:
from prepare_metadata import main as prepare_metadata

metadata_gt, metadata_predicted = prepare_metadata()
pd.Series({
    'metadata_phones_gt.csv': len(metadata_gt),
    'metadata_phones.csv': len(metadata_predicted),
}).to_frame('Строки')

,Строки
metadata_phones_gt.csv,19649
metadata_phones.csv,19649


## Результат

PER на test равен 0,0135. На словах вне обоих словарей модель получила 0,0916. Оба значения проходят заданные пороги 0,025 и 0,20.